### Phase 2: NLP Processing Pipeline

Enriches every raw post from Phase 1 with three layers of NLP analysis. Sentiment scoring classifies each post on a -1.0 to +1.0 scale using a social-media-trained RoBERTa model.

Stance detection uses zero-shot classification to determine whether the author is for, against, or neutral toward the topic — capturing nuance that sentiment alone misses.

Emotion classification breaks down each post into 7 emotions (anger, joy, fear, sadness, surprise, disgust, neutral), revealing why people feel the way they do.

On top of analysis, a sliding window algorithm detects opinion shifts in real time by comparing recent sentiment against a 6-hour baseline, and an argument clustering module extracts the dominant talking points per topic using TF-IDF. All results are stored back in Supabase and aggregated into 15-minute opinion snapshots for the dashboard.

This cell installs the necessary Python libraries for NLP processing, including `transformers`, `torch`, and `scipy`.

In [1]:
# Installing NLP dependencies
!pip install transformers torch scipy supabase

In [2]:
from collections import Counter, defaultdict
from datetime import datetime, timezone, timedelta

This section loads three pre-trained models from HuggingFace for sentiment analysis, emotion classification, and zero-shot stance detection. It also includes test cases to verify the models are working correctly.

In [3]:
## Loading Models
from transformers import pipeline

# ══════════════════════════════════════════
# MODEL 1: Sentiment Analysis
# ══════════════════════════════════════════
# Research this model card on HuggingFace:
# https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest
#
# Questions:
# - What labels does it output? (hint: not "positive"/"negative")
# - What does the score mean?
# - max_length matters — social posts are short, news articles aren't

print("Loading sentiment model...")
sentiment_model = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    truncation=True,
    max_length=512,
)

# ══════════════════════════════════════════
# MODEL 2: Emotion Classification
# ══════════════════════════════════════════
# Research: https://huggingface.co/j-hartmann/emotion-english-distilroberta-base
#
# Questions:
# - What 7 emotions does it detect?
# - What does top_k=None return vs top_k=1?
# - How big is this model compared to sentiment?

print("Loading emotion model...")
emotion_model = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None,      # Returns all 7 emotions with scores
    truncation=True,
    max_length=512,
)

# ══════════════════════════════════════════
# MODEL 3: Stance Detection (Zero-Shot)
# ══════════════════════════════════════════
# Research: https://huggingface.co/facebook/bart-large-mnli
#
# Questions:
# - What is zero-shot classification?
# - Why is this model much larger (~1.6GB)?
# - How slow is it compared to the other two?
# - What candidate_labels work best?

print("Loading stance model...")
stance_model = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    truncation=True,
)

print("All models loaded!")

# ── Test all three ──
test_text = "AI regulation is absolutely necessary to protect jobs and privacy"

print("\nSentiment:", sentiment_model(test_text))
print("\nEmotion:", emotion_model(test_text))
print("\nStance:", stance_model(
    test_text,
    candidate_labels=["in favor", "against", "neutral"],
))

Loading sentiment model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading emotion model...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading stance model...


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

All models loaded!

Sentiment: [{'label': 'neutral', 'score': 0.5387844443321228}]

Emotion: [[{'label': 'neutral', 'score': 0.6435526609420776}, {'label': 'fear', 'score': 0.22990405559539795}, {'label': 'anger', 'score': 0.05478941276669502}, {'label': 'disgust', 'score': 0.02470303513109684}, {'label': 'surprise', 'score': 0.01993531361222267}, {'label': 'joy', 'score': 0.015620414167642593}, {'label': 'sadness', 'score': 0.011495143175125122}]]

Stance: {'sequence': 'AI regulation is absolutely necessary to protect jobs and privacy', 'labels': ['in favor', 'against', 'neutral'], 'scores': [0.7371644973754883, 0.14981810748577118, 0.11301737278699875]}


In [4]:
TRACKED_TOPICS = {
    "artificial intelligence": [
        # Companies / models — specific enough standalone
        "openai", "chatgpt", "anthropic",
        "google gemini", "gemini ai",
        "gpt-4", "gpt-3", "gpt-4o",
        "github copilot", "microsoft copilot",
        "midjourney", "stable diffusion", "dall-e", "sora ai",
        "nvidia ai", "nvidia cuda",
        # Technical terms — compound required
        "large language model", "llm",
        "generative ai", "ai-generated",
        "foundation model", "diffusion model",
        "transformer model", "multimodal ai",
        "artificial general intelligence", "agi",
        "neural network architecture",
        # Policy / ethics — compound required
        "ai safety", "ai alignment",
        "ai ethics", "ai bias",
        "ai governance", "ai regulation",
        "eu ai act", "ai act",
        "ai deepfake", "deepfake detection",
        # Industry
        "ai chip", "ai startup", "ai investment",
        "artificial intelligence"
    ],

    "climate policy": [
        # Policy instruments — specific compound
        "climate policy", "climate legislation",
        "climate summit", "climate agreement",
        "paris agreement", "paris climate accord",
        "cop29", "cop28", "unfccc",
        "carbon tax", "carbon pricing",
        "carbon emissions target",
        "net zero target", "net zero commitment",
        "carbon neutral pledge",
        "renewable energy policy", "clean energy policy",
        "fossil fuel subsidy", "fossil fuel ban",
        "coal phase out", "oil phase out",
        "carbon capture technology",
        "greenhouse gas target",
        "green new deal",
        "climate refugee", "climate migration",
        "sea level rise",
        "deforestation policy", "amazon deforestation",
        "methane emissions target",
        "climate crisis", "climate emergency",
        "global warming",
        "carbon offset scheme"
    ],

    "data privacy": [
        # Regulation — specific
        "gdpr", "ccpa",
        "data privacy", "data protection law",
        "privacy law", "privacy regulation",
        "right to be forgotten",
        # Breaches — compound required
        "data breach", "personal data leak",
        "user data exposed", "data harvesting policy",
        # Surveillance — compound required
        "mass surveillance", "government surveillance",
        "surveillance capitalism",
        "facial recognition ban", "facial recognition law",
        "biometric surveillance",
        "location tracking ban", "phone tracking law",
        # Encryption / tools — specific
        "end-to-end encryption",
        "encrypted messaging",
        "pegasus spyware", "spyware attack",
        "cookie consent law", "online tracking ban",
        "vpn ban", "vpn law",
        "zero-knowledge proof"
    ],

    "tech layoffs": [
        "tech layoff", "tech layoffs",
        "technology layoff", "technology layoffs",
        "tech job cut", "tech job cuts",
        "tech worker laid off", "tech worker fired",
        "tech worker let go",
        "silicon valley layoff",
        "tech company cut",
        "software engineer laid off",
        "developer laid off", "engineer layoff",
        "startup layoff", "big tech layoff",
        "tech industry job loss",
        "tech hiring freeze",
        "tech workforce reduction",
        "google layoff", "amazon layoff",
        "meta layoff", "microsoft layoff",
        "apple layoff", "netflix layoff",
        "twitter layoff", "salesforce layoff",
        "intel layoff", "ibm layoff",
        "cisco layoff", "zoom layoff",
        "snap layoff", "lyft layoff",
        "uber layoff", "airbnb layoff",
        "stripe layoff"
    ],

    "business": [
        # Funding — compound required
        "series a funding", "series b funding",
        "series c funding", "seed funding round",
        "venture capital funding", "vc backed startup",
        "startup funding round",
        # M&A — compound required
        "company acquisition", "corporate acquisition",
        "hostile takeover", "company merger",
        "private equity buyout",
        # IPO — compound required
        "ipo filing", "ipo listing",
        "stock market debut", "going public ipo",
        # Earnings — compound required
        "quarterly earnings report", "earnings per share",
        "revenue guidance", "profit warning",
        "earnings beat", "earnings miss",
        # Antitrust — compound required
        "antitrust lawsuit", "antitrust investigation",
        "ftc lawsuit", "monopoly lawsuit",
        # Market / valuation — compound required
        "company valuation", "market valuation",
        "unicorn startup", "startup unicorn",
        "chapter 11 bankruptcy", "corporate bankruptcy",
        # Macro — compound required
        "supply chain disruption",
        "federal reserve rate", "interest rate hike",
        "inflation report", "inflation data"
    ],

    "us politics": [
        # Named leaders — specific enough standalone
        "donald trump", "jd vance", "bernie sanders",
        "pete buttigieg",
        "ron desantis",
        # Institutions + US qualifier — compound required
        "us congress", "us senate", "us house representatives",
        "white house", "oval office",
        "republican party", "democratic party",
        # Policy acts / events — compound or specific phrase
        "executive order",
        "supreme court ruling", "supreme court decision",
        "us midterm election", "presidential election",
        "us government shutdown", "debt ceiling crisis",
        "us tariff", "american tariff",
        "us immigration policy", "us border policy",
        "senate filibuster",
        "electoral college",
        "impeachment trial",
        "january 6 committee",
        "voter suppression law",
        "gerrymandering case",
        "us attorney general",
        "department of justice investigation",
        "us foreign policy", "us presidential race"
    ],

    "hollywood": [
        # Awards — specific
        "box office", "oscars", "academy awards",
        "oscar nomination", "oscar winner",
        "golden globes", "emmy awards", "bafta awards",
        # Industry disputes — specific
        "sag-aftra", "writers guild", "wga strike",
        "hollywood strike", "actors strike", "writers strike",
        # Festivals — compound required
        "cannes film festival", "sundance film festival",
        "toronto international film festival",
        # Industry metrics — compound required
        "box office record", "box office flop",
        "box office opening weekend",
        "movie premiere", "film release date",
        "film industry revenue", "rotten tomatoes score",
        # Platforms — compound required
        "netflix original series", "hbo original series",
        "disney plus original", "streaming rights deal",
        "paramount plus series", "peacock original",
        # Production news — compound required
        "casting announcement", "film director announced",
        "movie sequel announced", "film remake announced",
        "studio acquisition", "production deal"
    ],

    "world news": [
        # Specific active conflicts
        "israel hamas", "israel hamas war",
        "gaza ceasefire", "gaza war", "west bank settlement",
        "russia ukraine war", "ukraine war",
        "russia ukraine ceasefire",
        "taiwan strait tension", "taiwan china tension",
        "iran nuclear deal", "iran nuclear program",
        "north korea missile", "north korea nuclear",
        "south china sea dispute",
        # International bodies + action — compound required
        "united nations resolution", "un security council vote",
        "nato summit", "nato expansion",
        "european union summit", "eu parliament vote",
        "g7 summit", "g20 summit", "brics summit",
        # Diplomacy — compound required
        "ceasefire agreement", "peace negotiations",
        "international sanctions",
        # Crises — compound required
        "humanitarian crisis",
        "global refugee crisis",
        "global recession warning",
        "imf warning", "world bank report",
        "military offensive",
        "coup attempt",
        "terror attack",
        "assassination attempt",
        "nuclear treaty",
        "missile strike",
        "natural disaster death toll",
        "who declares health emergency",
        "disease outbreak warning"
    ],

    "tech news": [
        # Apple — product/event compound
        "apple iphone", "apple macbook", "apple mac",
        "apple vision pro", "apple watch", "apple tv",
        "apple earnings", "apple event", "wwdc",
        "apple silicon", "apple chip",
        # Google — product/event compound
        "google pixel", "google chrome update",
        "google search algorithm", "google earnings",
        "google io", "google cloud",
        # Microsoft — product/event compound
        "microsoft windows", "microsoft surface",
        "microsoft azure", "microsoft earnings",
        "microsoft teams update",
        # Amazon — tech compound
        "amazon aws", "amazon web services",
        "amazon alexa", "amazon echo",
        "amazon tech", "amazon earnings",
        # Meta — tech compound
        "meta quest", "meta earnings",
        "meta ai", "facebook algorithm",
        "instagram algorithm update",
        # Other hardware
        "samsung galaxy", "samsung chip",
        "intel chip", "amd chip",
        "qualcomm chip", "arm processor",
        # Industry events/topics — compound required
        "tech product launch", "tech conference",
        "cybersecurity breach", "ransomware attack",
        "semiconductor shortage", "chip shortage",
        "5g network rollout", "quantum computing breakthrough",
        "self-driving car", "autonomous vehicle",
        "tech antitrust", "app store policy",
        "data center investment"
    ]
}

This code block contains SQL commands to add new columns (`sentiment_score`, `sentiment_label`, `stance`, `emotions`, `is_processed`) to the `posts` table and create new tables (`opinion_snapshots`, `shift_events`) in Supabase. These columns and tables are used to store the NLP processing results and aggregated data.

In [5]:
import os
from google.colab import userdata
from supabase import create_client

# ══════════════════════════════════════════
# SUPABASE
# ══════════════════════════════════════════
SUPABASE_URL = userdata.get("Supabase_url")
SUPABASE_KEY = userdata.get("Supabase_Key")

# Run this once — adds NLP columns to your existing posts table
# Go to Supabase SQL Editor and run:

"""
ALTER TABLE posts
ADD COLUMN IF NOT EXISTS sentiment_score FLOAT,
ADD COLUMN IF NOT EXISTS sentiment_label TEXT,
ADD COLUMN IF NOT EXISTS stance TEXT,
ADD COLUMN IF NOT EXISTS emotions JSONB,
ADD COLUMN IF NOT EXISTS is_processed BOOLEAN DEFAULT false;

CREATE TABLE IF NOT EXISTS opinion_snapshots (
    id UUID DEFAULT gen_random_uuid() PRIMARY KEY,
    topic TEXT NOT NULL,
    window_start TIMESTAMPTZ NOT NULL,
    platform TEXT,
    avg_sentiment FLOAT,
    stance_distribution JSONB,
    emotion_distribution JSONB,
    post_count INT,
    top_arguments JSONB,
    UNIQUE(topic, window_start, platform)
);

CREATE TABLE IF NOT EXISTS shift_events (
    id UUID DEFAULT gen_random_uuid() PRIMARY KEY,
    topic TEXT NOT NULL,
    detected_at TIMESTAMPTZ DEFAULT now(),
    direction TEXT,
    magnitude FLOAT,
    trigger_posts JSONB,
    before_sentiment FLOAT,
    after_sentiment FLOAT
);

CREATE INDEX IF NOT EXISTS idx_snapshots_topic ON opinion_snapshots(topic);
CREATE INDEX IF NOT EXISTS idx_snapshots_window ON opinion_snapshots(window_start);
CREATE INDEX IF NOT EXISTS idx_shifts_topic ON shift_events(topic);
"""
# Initialize the client
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# After running the SQL, test it:
response = supabase.table("posts").select("sentiment_score,is_processed").limit(1).execute()
print("Schema updated!" if response else "Check your SQL")

Schema updated!


This cell defines three helper functions: `analyze_sentiment` (to get sentiment score and label), `analyze_emotions` (to classify emotions), and `analyze_stance` (to determine the author's stance on a topic). Each function includes logic for handling edge cases and converting model outputs to desired formats.

In [6]:
def analyze_sentiment(text):
    """
    Returns: (score: float, label: str)

    score range: -1.0 to +1.0
    label: "positive" / "negative" / "neutral"

    Think about:
    - The model outputs labels like "positive", "negative", "neutral"
      BUT check the model card — it might use different label names
    - How do you convert model output to a -1 to +1 scale?
      negative → -1 × confidence
      neutral → 0
      positive → +1 × confidence
    - What if text is empty?
    - What if text is super long? (truncation handles it)
    """
    if not text or len(text.strip()) < 5:
        return 0.0, "neutral"

    try:
        result = sentiment_model(text)[0]
        label = result["label"].lower()
        score = result["score"]

        # Map to -1 to +1 scale
        # Research: what exact labels does this model use?
        # It might be "positive"/"negative"/"neutral"
        # or it might be "LABEL_0"/"LABEL_1"/"LABEL_2"
        # Check the model card!

        if "negative" in label or label == "negative":
            return round(-1 * score, 4), "negative"
        elif "positive" in label or label == "positive":
            return round(score, 4), "positive"
        else:
            return 0.0, "neutral"
    except Exception as e:
        print(f"Sentiment error: {e}")
        return 0.0, "neutral"


def analyze_emotions(text):
    """
    Returns: dict like {"anger": 0.7, "joy": 0.05, "fear": 0.1, ...}
    """
    if not text or len(text.strip()) < 5:
        return {"neutral": 1.0}

    try:
        # The model with top_k=None returns a NESTED list:
        # [[{"label": "anger", "score": 0.7}, {"label": "joy", "score": 0.1}, ...]]
        results = emotion_model(text)

        # Unwrap the outer list
        if isinstance(results, list) and len(results) > 0:
            if isinstance(results[0], list):
                results = results[0]  # Nested case

        # Convert list of {label, score} dicts to flat dict
        emotions_dict = {}
        for item in results:
            if isinstance(item, dict) and "label" in item and "score" in item:
                emotions_dict[item["label"]] = round(item["score"], 3)

        if not emotions_dict:
            return {"neutral": 1.0}

        return emotions_dict

    except Exception as e:
        print(f"Emotion error: {e}")
        return {"neutral": 1.0}


def analyze_stance(text, topic):
    """
    Returns: "for" / "against" / "neutral"

    Think about:
    - candidate_labels: try different phrasings
      Option A: ["in favor", "against", "neutral"]
      Option B: ["supports {topic}", "opposes {topic}", "neutral about {topic}"]
      Which works better? Test both!
    - Confidence threshold: if top score < 0.45, call it "neutral"
      Why? Because 33% is random chance for 3 labels
    - This model is SLOW (~0.5 sec/post on CPU)
      Skip for posts with weak sentiment? (abs(score) < 0.3)
    """
    if not text or len(text.strip()) < 10:
        return "neutral"

    try:
        result = stance_model(
            text,
            candidate_labels=["in favor", "against", "neutral"],
        )

        top_label = result["labels"][0]
        top_score = result["scores"][0]

        # If confidence is too low, default to neutral
        if top_score < 0.40:
            return "neutral"

        # Map labels to standard values
        if "favor" in top_label or "support" in top_label:
            return "for"
        elif "against" in top_label or "oppose" in top_label:
            return "against"
        else:
            return "neutral"

    except Exception as e:
        print(f"Stance error: {e}")
        return "neutral"

This `process_batch` function takes a list of raw posts, iterates through them, and applies the sentiment, emotion, and stance analysis models. It handles potential errors for individual posts and allows for skipping stance analysis on posts with weak sentiment to optimize performance.

In [7]:
def process_batch(posts):
    """
    Run all 3 models on a list of posts

    Think about:
    - Sentiment + Emotion are fast (~50 posts/sec)
    - Stance is SLOW (~2 posts/sec) — it's the bottleneck
    - Skip stance for weak sentiment posts to save time
    - Show progress every 10 posts
    - If one post crashes a model, skip it, don't kill the batch
    """
    processed = []

    for i, post in enumerate(posts):
        try:
            content = post.get("content", "")
            topic = post.get("topic", "")

            # 1. Sentiment (fast)
            score, label = analyze_sentiment(content)

            # 2. Emotion (fast)
            emotions = analyze_emotions(content)

            # 3. Stance (slow — skip for weak sentiment)
            if abs(score) > 0.3:
                stance = analyze_stance(content, topic)
            else:
                stance = "neutral"

            # 4. Build update payload
            post["sentiment_score"] = score
            post["sentiment_label"] = label
            post["emotions"] = emotions
            post["stance"] = stance
            post["is_processed"] = True

            processed.append(post)

            # 5. Progress
            if (i + 1) % 10 == 0:
                print(f"  Processed {i+1}/{len(posts)}")

        except Exception as e:
            print(f"  Error on post {post.get('platform_id', '?')}: {e}")
            continue

    return processed

This `save_nlp_results` function is responsible for updating the Supabase `posts` table with the NLP analysis results (sentiment score, label, stance, emotions, and processing status) for a batch of processed posts.

In [8]:
def save_nlp_results(client, processed_posts):
    """
    Update existing posts with NLP columns

    Think about:
    - You're UPDATING existing rows, not inserting new ones
    - Update by matching on (platform, platform_id)
    - Only send the NLP columns, not the full post
    - Batch or one by one?
      Supabase upsert can handle batches
    """
    if not processed_posts:
        return 0

    updates = []
    for post in processed_posts:
        updates.append({
            "platform": post["platform"],
            "platform_id": post["platform_id"],
            "sentiment_score": post["sentiment_score"],
            "sentiment_label": post["sentiment_label"],
            "stance": post["stance"],
            "emotions": post["emotions"],
            "is_processed": True,
        })

    try:
        # Upsert using the same conflict columns
        response = client.table("posts").upsert(
            updates, on_conflict="platform,platform_id"
        ).execute()
        return len(updates)
    except Exception as e:
        print(f"Error saving NLP results: {e}")
        return 0

This `detect_shifts` function implements a sliding window algorithm to compare recent sentiment against a baseline. It fetches processed posts, calculates mean sentiment, and detects significant deviations, storing any detected shifts in the `shift_events` table.

In [9]:
## Shift detection Algorithm
import numpy as np

def detect_shifts(client, topic, short_hours=1, long_hours=6, threshold=2.0):
    """
    Compare recent sentiment against baseline

    Algorithm:
    1. Fetch posts from last 6 hours for this topic
    2. Split into baseline (older) and recent (last 1 hour)
    3. If recent mean deviates by > 2 std from baseline → SHIFT

    Questions:
    - What if < 10 posts in either window? (skip — not enough data)
    - What if baseline std is 0? (all same sentiment — skip)
    - How do you find the "trigger posts"?
    """
    try:
        now = datetime.now(timezone.utc)
        long_cutoff = (now - timedelta(hours=long_hours)).isoformat()
        short_cutoff = (now - timedelta(hours=short_hours)).isoformat()

        # Fetch all processed posts for this topic in long window
        response = client.table("posts").select(
            "platform_id,sentiment_score,created_at"
        ).eq("topic", topic).eq(
            "is_processed", True
        ).gte("created_at", long_cutoff).execute()

        all_posts = response.data if response.data else []

        if len(all_posts) < 15:
            return False  # Not enough data

        # Split into baseline and recent
        baseline = [p for p in all_posts if p["created_at"] < short_cutoff]
        recent = [p for p in all_posts if p["created_at"] >= short_cutoff]

        if len(baseline) < 10 or len(recent) < 5:
            return False

        baseline_scores = [p["sentiment_score"] for p in baseline]
        recent_scores = [p["sentiment_score"] for p in recent]

        baseline_mean = np.mean(baseline_scores)
        baseline_std = np.std(baseline_scores)
        recent_mean = np.mean(recent_scores)

        if baseline_std == 0:
            return False

        deviation = abs(recent_mean - baseline_mean)

        if deviation > threshold * baseline_std:
            # SHIFT DETECTED
            direction = "positive" if recent_mean > baseline_mean else "negative"
            magnitude = round(deviation / baseline_std, 2)

            # Find trigger posts (most extreme in recent window)
            recent_sorted = sorted(recent, key=lambda p: abs(p["sentiment_score"] - baseline_mean), reverse=True)
            triggers = [p["platform_id"] for p in recent_sorted[:5]]

            # Store shift event
            shift = {
                "topic": topic,
                "direction": direction,
                "magnitude": magnitude,
                "trigger_posts": triggers,
                "before_sentiment": round(baseline_mean, 4),
                "after_sentiment": round(recent_mean, 4),
            }
            client.table("shift_events").insert(shift).execute()
            print(f"  ⚠ SHIFT DETECTED in '{topic}': {direction} (magnitude: {magnitude})")
            return True

        return False

    except Exception as e:
        print(f"  Shift detection error for '{topic}': {e}")
        return False

This `cluster_arguments` function uses TF-IDF (Term Frequency-Inverse Document Frequency) to extract dominant talking points or arguments from a collection of posts. It identifies the most important terms and n-grams to represent the key themes.

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

def cluster_arguments(posts, min_posts=5):
    """
    Group posts into talking point clusters

    Questions:
    - What's ngram_range? Why (1,3)?
    - max_features limits vocabulary size — why 500?
    - How do you label each cluster? (top TF-IDF terms)
    - HDBSCAN vs simpler approach?

    For now: use a simpler approach — extract top keyword groups
    You can upgrade to HDBSCAN later
    """
    if len(posts) < min_posts:
        return []

    texts = [p["content"] for p in posts if p.get("content")]

    if len(texts) < min_posts:
        return []

    try:
        vectorizer = TfidfVectorizer(
            max_features=500,
            stop_words="english",
            ngram_range=(1, 3),
        )
        tfidf_matrix = vectorizer.fit_transform(texts)
        feature_names = vectorizer.get_feature_names_out()

        # Simple approach: top N terms across all posts as "arguments"
        # Sum TF-IDF scores per term across all posts
        term_scores = tfidf_matrix.sum(axis=0).A1
        top_indices = term_scores.argsort()[-10:][::-1]

        arguments = []
        for idx in top_indices:
            arguments.append({
                "label": feature_names[idx],
                "score": round(float(term_scores[idx]), 3),
            })

        return arguments

    except Exception as e:
        print(f"Clustering error: {e}")
        return []

This `create_snapshots` function aggregates processed posts into 15-minute opinion snapshots. It calculates average sentiment, stance distribution, emotion distribution, post count, and top arguments per topic and platform, storing these aggregations in the `opinion_snapshots` table.

In [11]:
def create_snapshots(client, topics):
    """
    Aggregate processed posts into opinion snapshots

    Think about:
    - Round current time down to nearest 15 min
    - Group by topic + platform
    - Calculate avg sentiment, stance distribution, emotion distribution
    """
    now = datetime.now(timezone.utc)
    minutes = now.minute - (now.minute % 15)
    window_start = now.replace(minute=minutes, second=0, microsecond=0)
    window_end = window_start + timedelta(minutes=15)

    for topic in topics:
        try:
            # Fetch processed posts in this window
            response = client.table("posts").select(
                "source_name,sentiment_score,stance,emotions"
            ).eq("topic", topic).eq(
                "is_processed", True
            ).gte("created_at", window_start.isoformat()
            ).lt("created_at", window_end.isoformat()).execute()

            posts = response.data if response.data else []

            if not posts:
                continue

            # Group by platform
            from collections import defaultdict
            by_platform = defaultdict(list)
            for p in posts:
                by_platform[p["source_name"]].append(p)

            for platform, platform_posts in by_platform.items():
                # Calculate aggregations
                scores = [p["sentiment_score"] for p in platform_posts if p.get("sentiment_score") is not None]
                avg_sent = round(np.mean(scores), 4) if scores else 0

                # Stance distribution
                stances = [p["stance"] for p in platform_posts if p.get("stance")]
                stance_dist = dict(Counter(stances))

                # Emotion distribution (average each emotion)
                # Your logic — average the emotion dicts across posts
                emotion_sums = {}
                emotion_count = 0
                for p in platform_posts:
                    if p.get("emotions"):
                        emotions = p["emotions"]
                        if isinstance(emotions, str):
                            import json
                            emotions = json.loads(emotions)
                        for emo, val in emotions.items():
                            emotion_sums[emo] = emotion_sums.get(emo, 0) + val
                        emotion_count += 1

                emotion_dist = {}
                if emotion_count > 0:
                    emotion_dist = {k: round(v/emotion_count, 3) for k, v in emotion_sums.items()}

                # Arguments
                arguments = cluster_arguments(platform_posts)

                snapshot = {
                    "topic": topic,
                    "window_start": window_start.isoformat(),
                    "platform": platform,
                    "avg_sentiment": avg_sent,
                    "stance_distribution": stance_dist,
                    "emotion_distribution": emotion_dist,
                    "post_count": len(platform_posts),
                    "top_arguments": arguments,
                }

                client.table("opinion_snapshots").upsert(
                    snapshot, on_conflict="topic,window_start,platform"
                ).execute()

        except Exception as e:
            print(f"Snapshot error for '{topic}': {e}")

This `run_nlp_pipeline` function orchestrates the entire NLP processing workflow. It fetches unprocessed posts, applies NLP models, saves results, detects opinion shifts, and creates opinion snapshots. It also provides a summary of the processing.

In [22]:
def run_nlp_pipeline(max_posts=400):
    """
    Fetch unprocessed posts → analyze → update → detect shifts → snapshot
    """
    print("=" * 60)
    print("  NLP PIPELINE — Phase 2")
    print("=" * 60)

    # 1. Fetch unprocessed posts
    print("\nFetching unprocessed posts...")
    response = supabase.table("posts").select("*").eq(
        "is_processed", False
    ).limit(max_posts).execute()

    posts = response.data if response.data else []
    print(f"Found {len(posts)} unprocessed posts")

    if not posts:
        print("Nothing to process!")
        return

    # 2. Process
    print("\nRunning NLP analysis...")
    processed = process_batch(posts)
    print(f"\nSuccessfully processed: {len(processed)}/{len(posts)}")

    # 3. Save results
    print("\nSaving to Supabase...")
    saved = save_nlp_results(supabase, processed)
    print(f"Updated {saved} posts with NLP data")

    # 4. Shift detection
    print("\nChecking for opinion shifts...")
    for topic in TRACKED_TOPICS:
        detect_shifts(supabase, topic)

    # 5. Create snapshots
    print("\nCreating opinion snapshots...")
    create_snapshots(supabase, TRACKED_TOPICS)

    # 6. Summary
    print(f"\n{'='*60}")
    print("  NLP PIPELINE COMPLETE")
    print(f"  Posts processed: {len(processed)}")

    # Quick stats
    sentiments = Counter(p["sentiment_label"] for p in processed)
    stances = Counter(p["stance"] for p in processed)
    print(f"  Sentiment: {dict(sentiments)}")
    print(f"  Stance: {dict(stances)}")
    print(f"{'='*60}")

run_nlp_pipeline()

  NLP PIPELINE — Phase 2

Fetching unprocessed posts...
Found 1 unprocessed posts

Running NLP analysis...

Successfully processed: 1/1

Saving to Supabase...
Updated 1 posts with NLP data

Checking for opinion shifts...

Creating opinion snapshots...

  NLP PIPELINE COMPLETE
  Posts processed: 1
  Sentiment: {'positive': 1}
  Stance: {'for': 1}


This `verify_phase2` function provides a detailed verification of the NLP processing. It checks the counts of processed vs. unprocessed posts, displays sentiment and stance distributions, calculates average sentiment per topic, and reports on detected shift events and created opinion snapshots.

In [23]:
def verify_phase2():
    print("\n--- Phase 2 Verification ---")

    # Processed vs unprocessed
    processed = supabase.table("posts").select("*").eq("is_processed", True).execute()
    unprocessed = supabase.table("posts").select("*").eq("is_processed", False).execute()

    p_count = len(processed.data) if processed.data else 0
    u_count = len(unprocessed.data) if unprocessed.data else 0
    print(f"\n1. Processed: {p_count} | Unprocessed: {u_count}")

    if processed.data:
        posts = processed.data

        # Sentiment distribution
        print("\n2. Sentiment Distribution:")
        sent_counts = Counter(p["sentiment_label"] for p in posts)
        for label, count in sent_counts.most_common():
            pct = round(count / len(posts) * 100, 1)
            print(f"  {label}: {count} ({pct}%)")

        # Stance distribution
        print("\n3. Stance Distribution:")
        stance_counts = Counter(p["stance"] for p in posts)
        for stance, count in stance_counts.most_common():
            pct = round(count / len(posts) * 100, 1)
            print(f"  {stance}: {count} ({pct}%)")

        # Sentiment per topic
        print("\n4. Avg Sentiment per Topic:")
        from collections import defaultdict
        topic_scores = defaultdict(list)
        for p in posts:
            if p.get("sentiment_score") is not None:
                topic_scores[p["topic"]].append(p["sentiment_score"])
        for topic, scores in sorted(topic_scores.items()):
            avg = round(np.mean(scores), 3)
            print(f"  {topic}: {avg} ({len(scores)} posts)")

        # Shift events
        shifts = supabase.table("shift_events").select("*").execute()
        s_count = len(shifts.data) if shifts.data else 0
        print(f"\n5. Shift Events Detected: {s_count}")
        if shifts.data:
            for s in shifts.data:
                print(f"  {s['topic']}: {s['direction']} (magnitude: {s['magnitude']})")

        # Snapshots
        snaps = supabase.table("opinion_snapshots").select("*").execute()
        snap_count = len(snaps.data) if snaps.data else 0
        print(f"\n6. Opinion Snapshots: {snap_count}")

    print("----------------------------")

verify_phase2()


--- Phase 2 Verification ---

1. Processed: 1000 | Unprocessed: 0

2. Sentiment Distribution:
  neutral: 462 (46.2%)
  negative: 370 (37.0%)
  positive: 168 (16.8%)

3. Stance Distribution:
  neutral: 472 (47.2%)
  against: 367 (36.7%)
  for: 161 (16.1%)

4. Avg Sentiment per Topic:
  artificial intelligence: -0.068 (103 posts)
  business: 0.184 (14 posts)
  climate policy: -0.263 (43 posts)
  data privacy: -0.213 (10 posts)
  hollywood: -0.047 (88 posts)
  other: -0.128 (555 posts)
  tech layoffs: -0.445 (10 posts)
  tech news: -0.037 (34 posts)
  us politics: -0.288 (99 posts)
  world news: -0.329 (44 posts)

5. Shift Events Detected: 0

6. Opinion Snapshots: 0
----------------------------
